In [1]:
# Cell 1: Environment and Auth
!pip install -q -U transformers accelerate bitsandbytes

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

print("Logging into Hugging Face...")
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 59.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 87.6 MB/s eta 0:00:00:00:01
Logging into Hugging Face...
✅ Ready.


In [3]:
# Cell 2: Model & Data Loading
import json
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

INPUT_FILE = "/kaggle/input/datasets/armanhalavi/thesis-clean-text/clean_human_paragraphs.json"
OUTPUT_FILE = "/kaggle/working/paired_training_data.jsonl"
MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print("Loading 4-bit Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Llama-3-Instruct...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    human_paragraphs = json.load(f)

print(f"✅ Model loaded and {len(human_paragraphs)} paragraphs found.")

Loading 4-bit Quantization...
Loading Llama-3-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded and 2887 paragraphs found.


In [4]:
# Cell 3: The Interactive Sample Test
sample_text = human_paragraphs[0] # Change the 0 to any number to test different paragraphs

print("--- ORIGINAL HUMAN TEXT ---")
print(sample_text)
print("\n--- GENERATING AI DRAFT... ---\n")

messages = [
    {"role": "system", "content": "You are a standard, helpful AI assistant. Rewrite the user's text in a clear, polite, and standard AI tone. Do not use overly complex academic jargon, but maintain the core facts. Return ONLY the rewritten text."},
    {"role": "user", "content": sample_text}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=800, 
        temperature=0.4, 
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True
    )

input_length = inputs["input_ids"].shape[1]
response_ids = outputs[0][input_length:]
ai_draft = tokenizer.decode(response_ids, skip_special_tokens=True).strip()

print("--- SYNTHETIC AI TEXT ---")
print(ai_draft)

Both `max_new_tokens` (=800) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- ORIGINAL HUMAN TEXT ---
This paper uses fractional integration methods to analyze persistence in the five cryptocurrencies with the highest degree of market capitalization (Bitcoin, Ethereum, Tether, BNB and USD Coin), and also the possible impact of both the Covid-19 pandemic and the Russia-Ukraine war. The findings suggest mean reversion in the case of the USD Coin, an explosive pattern in the case of BNB, and anti-persistence in the case of Tether. The unit root null hypothesis cannot be rejected in the case of Bitcoin, Ethereum and USD Coin, whilst it is rejected in favor of d > 1 in the case of BNB (where d is the fractional differencing parameter measuring persistence). Finally, there is not much evidence that either the Covid-19 pandemic or the Russia-Ukraine conflict have significantly affected persistence in the cryptocurrency market.

--- GENERATING AI DRAFT... ---

--- SYNTHETIC AI TEXT ---
Here is a rewritten version of the text in a clear, polite, and standard AI tone:

In [ ]:
# Cell 4: The Main Generation Loop (Resume-Safe / Append Mode)
import time

start_index = 0
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        start_index = sum(1 for _ in f)
    print(f"\n✅ Found existing file! Resuming from paragraph {start_index}...")
else:
    print(f"\nStarting fresh generation for {len(human_paragraphs)} paragraphs...")

start_time = time.time()

for i, human_text in enumerate(human_paragraphs):
    
    if i < start_index:
        continue
    
    messages = [
        {"role": "system", "content": "You are a standard, helpful AI assistant. Rewrite the user's text in a clear, polite, and standard AI tone. Do not use overly complex academic jargon, but maintain the core facts. Return ONLY the rewritten text."},
        {"role": "user", "content": human_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800, 
            temperature=0.4, 
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True
        )
    
    input_length = inputs["input_ids"].shape[1]
    response_ids = outputs[0][input_length:]
    ai_draft = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    
    formatted_pair = {
        "messages": [
            {
                "role": "system", 
                "content": "You are a direct, minimalist academic researcher in statistics and data science. Your sole job is to rewrite AI-generated drafts into dry, human-sounding academic prose. Preserve the exact length and paragraph structure of the input. Do not use flowery language."
            },
            {
                "role": "user", 
                "content": f"Rewrite this:\n\n{ai_draft}"
            },
            {
                "role": "assistant", 
                "content": human_text 
            }
        ]
    }
    
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(formatted_pair) + "\n")
    
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(human_paragraphs)} pairs...")

end_time = time.time()
print(f"\n✅ Finished in {round((end_time - start_time) / 60, 2)} minutes.")